<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Bar Charts**


Estimated time needed: **30** minutes


In this lab, you will focus on visualizing data.

The dataset will be provided to you in the form of an RDBMS.

You will use SQL queries to extract the necessary data.


## Objectives


In this lab you will perform the following:


-   Visualize the distribution of data

-   Visualize the relationship between two features

-   Visualize the composition of data

-   Visualize comparison of data


## Setup: Working with the Database
**Install the needed libraries**


In [ ]:
!pip install pandas

In [ ]:
!pip install matplotlib

**Download and connect to the database file containing survey data.**


To start, download and load the dataset into a `pandas` DataFrame.



In [ ]:
# Step 1: Download the dataset
!wget -O survey-data.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

# Step 2: Import necessary libraries and load the dataset
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("survey-data.csv")

# Display the first few rows to understand the structure of the data
df.head()


### Task 1: Visualizing Data Distributions


##### 1. Histogram of `ConvertedCompYearly`


Visualize the distribution of yearly compensation (`ConvertedCompYearly`) using a histogram.



In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

for col in ['ConvertedCompYearly', 'JobSatPoints_6', 'JobSatPoints_7']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
comp_limit = df['ConvertedCompYearly'].quantile(0.99)

# ---- 1.1 Histogram of ConvertedCompYearly ----
plt.figure(figsize=(10, 5))
plt.hist(df.loc[df['ConvertedCompYearly'] <= comp_limit, 'ConvertedCompYearly'], bins=50,
         color='steelblue', edgecolor='black')
plt.title('Distribution of Yearly Compensation (up to 99th percentile)')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('Number of respondents')
plt.show()

##### 2. Box Plot of `Age`


Since `Age` is categorical in the dataset, convert it to numerical values for a box plot.



In [ ]:
# ---- 1.2 Box plot of Age (converted to numbers) ----
df['Age_numeric'] = df['Age'].map(AGE_MAP)

plt.figure(figsize=(6, 7))
plt.boxplot(df['Age_numeric'].dropna())
plt.title('Box Plot of Age')
plt.ylabel('Age (approximate, years)')
plt.xticks([1], ['Age_numeric'])
plt.show()

### Task 2: Visualizing Relationships in Data


##### 1. Scatter Plot of `Age_numeric` and `ConvertedCompYearly`


Explore the relationship between age and compensation.



In [ ]:
# ---- 2.1 Scatter plot of Age_numeric vs ConvertedCompYearly ----
d = df[['Age_numeric', 'ConvertedCompYearly']].dropna()
d = d[d['ConvertedCompYearly'] <= comp_limit]

plt.figure(figsize=(10, 6))
plt.scatter(d['Age_numeric'] + np.random.uniform(-2, 2, len(d)), d['ConvertedCompYearly'], alpha=0.2)
plt.title('Age vs Yearly Compensation')
plt.xlabel('Age (approximate)')
plt.ylabel('ConvertedCompYearly (USD)')
plt.show()

##### 2. Bubble Plot of `ConvertedCompYearly` and `JobSatPoints_6` with `Age_numeric` as Bubble Size


Explore how compensation and job satisfaction are related, with age as the bubble size.


In [ ]:
# ---- 2.2 Bubble plot: ConvertedCompYearly vs JobSatPoints_6, bubble size = Age_numeric ----
d = df[['ConvertedCompYearly', 'JobSatPoints_6', 'Age_numeric']].dropna()
d = d[d['ConvertedCompYearly'] <= comp_limit]

plt.figure(figsize=(12, 7))
plt.scatter(d['ConvertedCompYearly'], d['JobSatPoints_6'], s=d['Age_numeric'] * 2,
            alpha=0.3, color='seagreen', edgecolors='w', linewidth=0.5)
plt.title('Compensation vs Job Satisfaction (bubble size = age)')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('JobSatPoints_6')
plt.show()

### Task 3: Visualizing Composition of Data with Bar Charts


##### 1. Horizontal Bar Chart of `MainBranch` Distribution


Visualize the distribution of respondents’ primary roles to understand their professional focus.



In [ ]:
# ---- 3.1 Horizontal bar chart of MainBranch ----
mb = df['MainBranch'].value_counts().sort_values()

plt.figure(figsize=(10, 5))
plt.barh(mb.index, mb.values, color='steelblue')
plt.title("Respondents' Main Branch")
plt.xlabel('Number of respondents')
plt.tight_layout()
plt.show()

##### 2. Vertical Bar Chart of Top 5 Programming Languages Respondents Want to Work With


Identify the most desired programming languages based on `LanguageWantToWorkWith`.



In [ ]:
# ---- 3.2 Top 5 languages respondents want to work with ----
top5 = split_counts(df['LanguageWantToWorkWith'], 5)

plt.figure(figsize=(10, 5))
plt.bar(top5.index, top5.values, color='darkorange', edgecolor='black')
plt.title('Top 5 Programming Languages Respondents Want to Work With')
plt.xlabel('Language')
plt.ylabel('Number of respondents')
plt.show()

##### 3. Stacked Bar Chart of Median `JobSatPoints_6` and `JobSatPoints_7` by Age Group


Compare job satisfaction metrics across different age groups with a stacked bar chart.


In [ ]:
# ---- 3.3 Stacked bar of median JobSatPoints_6 and JobSatPoints_7 by age group ----
by_age = df.groupby('Age')[['JobSatPoints_6', 'JobSatPoints_7']].median()
by_age = by_age.reindex([a for a in AGE_ORDER if a in by_age.index])

by_age.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Median JobSatPoints_6 and JobSatPoints_7 by Age Group')
plt.xlabel('Age group')
plt.ylabel('Median points')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

##### 4. Bar Chart of Database Popularity (`DatabaseHaveWorkedWith`)


Identify the most commonly used databases among respondents by visualizing `DatabaseHaveWorkedWith`.



In [ ]:
# ---- 3.4 Database popularity (DatabaseHaveWorkedWith) ----
top_db = split_counts(df['DatabaseHaveWorkedWith'], 10)

plt.figure(figsize=(12, 6))
plt.bar(top_db.index, top_db.values, color='purple', edgecolor='black')
plt.title('Top 10 Databases Respondents Have Worked With')
plt.xlabel('Database')
plt.ylabel('Number of respondents')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Task 4: Visualizing Comparison of Data with Bar Charts


##### 1. Grouped Bar Chart of Median `ConvertedCompYearly` for Different Age Groups


Compare median compensation across multiple age groups with a grouped bar chart.



In [ ]:
# ---- 4.1 Grouped bar chart of median ConvertedCompYearly by age group ----
# Grouped by the three most common employment types so each age group has bars side by side
top_emp = df['Employment'].value_counts().head(3).index
grouped = (df[df['Employment'].isin(top_emp)]
           .groupby(['Age', 'Employment'])['ConvertedCompYearly'].median().unstack())
grouped = grouped.reindex([a for a in AGE_ORDER if a in grouped.index])
print(grouped)

grouped.plot(kind='bar', figsize=(13, 6))
plt.title('Median Yearly Compensation by Age Group and Employment Type')
plt.xlabel('Age group')
plt.ylabel('Median ConvertedCompYearly (USD)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Employment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

##### 2. Bar Chart of Respondent Count by Country


Show the distribution of respondents by country to see which regions are most represented.



In [ ]:
# ---- 4.2 Respondent count by country (top 15) ----
countries = df['Country'].value_counts().head(15).sort_values()

plt.figure(figsize=(10, 7))
plt.barh(countries.index, countries.values, color='steelblue')
plt.title('Respondents by Country (top 15)')
plt.xlabel('Number of respondents')
plt.tight_layout()
plt.show()

### Final Step: Review


This lab demonstrates how to create and interpret different types of bar charts, allowing you to analyze the composition, comparison, and distribution of categorical data in the Stack Overflow dataset, including main professional branches, programming language preferences, and compensation by age group. Bar charts effectively compare counts and median values across various categories.


## Summary


After completing this lab, you will be able to:
- Create a horizontal bar chart to visualize the distribution of respondents' primary roles, helping to understand their professional focus.
- Develop a vertical bar chart to identify the most desired programming languages based on the LanguageWantToWorkWith variable.
- Use a stacked bar chart to compare job satisfaction metrics across different age groups.
- Create a bar chart to visualize the most commonly used databases among respondents using the DatabaseHaveWorkedWith variable.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
